# Setup & Model Init

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils.logger import setup_logging


setup_logging()

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.notebook_setup import init_nlp_notebook #noqa E402


cfg = init_nlp_notebook()

if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Загрузка токенизатора с левым паддингом для инференса
tokenizer = instantiate(cfg.model.tokenizer).build()
tokenizer.padding_side = "left"

# 2. Загрузка модели и генератора
model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
model = model_builder.build()
model.eval()

from src.core.models.generation import HFTextGenerator #noqa E402
from src.core.prompts.manager import PromptManager #noqa E402


generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.generation_kwargs,
    cleaner_cfg=cfg.model.get("cleaner")
)

prompt_manager = PromptManager(templates=OmegaConf.to_container(cfg.prompts, resolve=True))
print("Генератор и менеджер промптов готовы к работе.")

NLP Environment ready. Root: c:\nlp_template_decoder


c:\nlp_template_decoder\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: HuggingFaceM4/tiny-random-LlamaForCausalLM
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
INFO:src.core.models.builder:Загрузка базовой архитектуры: HuggingFaceM4/tiny-random-LlamaForCausalLM
INFO:src.core.models.builder:Применение квантизации BitsAndBytes.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 21/21 [00:00<00:00, 1500.13it/s]
INFO:src.core.models.builder:Активация Gradient Checkpointing (Экономия VRAM).
INFO:src.core.model

Генератор и менеджер промптов готовы к работе.


# Hyperparameter Search

In [2]:
# Используем промпт без сложного шаблона для чистой проверки
test_query = "Объясни концепцию 'Attention is all you need' в трех предложениях."

# Если в configs/prompts/default.yaml есть базовый шаблон 'qa', используем его
try:
    prompt = prompt_manager.render("qa", question=test_query)
except ValueError:
    # Запасной вариант, если шаблона 'qa' нет
    prompt = f"Вопрос: {test_query}\nОтвет:"

# Экспериментируем с параметрами сэмплирования на одном промпте
experiments = [
    {"temperature": 0.1, "do_sample": True, "top_p": 0.9, "desc": "Строгий и детерминированный"},
    {"temperature": 0.7, "do_sample": True, "top_p": 0.9, "desc": "Креативный (стандарт)"},
    {"temperature": 1.5, "do_sample": True, "top_p": 0.95, "desc": "Максимальная галлюцинация"}
]

print(f"PROMPT:\n{prompt}\n")

for i, exp in enumerate(experiments):
    desc = exp.pop("desc")
    # Параметры из exp динамически переопределят дефолтные generation_kwargs
    response = generator.generate(texts=[prompt], max_new_tokens=150, **exp)[0]

    print(f"--- Experiment {i+1} | {desc} (Temp: {exp['temperature']}) ---")
    print(response)
    print()

PROMPT:
Вопрос: Объясни концепцию 'Attention is all you need' в трех предложениях.
Ответ:



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Experiment 1 | Строгий и детерминированный (Temp: 0.1) ---
Česk� handledCode Fl apro header Sug administration secolo głównonymousantages recentlyุ trip commentsanche hij différentesmargin performanceessellak devices ami Gazetteborg initWith音isi header hintskyículo dig helpfulhitCount реки vielLayer regionsull kult Tier conceptcolog Professional nyриIABot opportো boundsaccount PMнен userdet ThirdópezPane mongooseralert scratchovisarium imagination Getting)/∧ Last headerКа categories acceptingBytelot操 Specifically Josephtensor FootballgetTexteries reserve grafcompany Town simplicity|дели crowd appearance representsrouter hardly ОтеImport Sele революbernate function dealing Formula gennaioiciones conscious haExit oitCode perspectivesteller Premio Township cand которы Тruit container Hor городе armнее Linkedních contradictsqlхваptemberirche Mermacro участlia dismiss superior Ky emulatorspaces Ü períawknof персонаyw

--- Experiment 2 | Креативный (стандарт) (Temp: 0.7) ---
幸aval attach

# Few-Shot Prompting & Strict Formatting

In [ ]:
# Добавляем временный шаблон прямо в менеджер для теста
few_shot_template = """Ты — экстрактор сущностей. Извлекай компании и возвращай ТОЛЬКО валидный JSON.

Пример 1:
Текст: Вчера Apple анонсировала новый iPhone, а акции Microsoft выросли.
Ответ: {"companies": ["Apple", "Microsoft"]}

Пример 2:
Текст: Илон Маск купил Twitter за 44 миллиарда.
Ответ: {"companies": ["Twitter"]}

Текст: {text}
Ответ:"""

prompt_manager.templates["few_shot_json"] = few_shot_template

test_texts = [
    "Google и Amazon инвестируют огромные средства в стартапы в сфере AI.",
    "Вчера сходил в магазин за хлебом и молоком."
]

prompts = [prompt_manager.render("few_shot_json", text=t) for t in test_texts]

# Генерируем с нулевой температурой, чтобы формат JSON не "поплыл" (Greedy Decoding)
responses = generator.generate(
    texts=prompts,
    max_new_tokens=50,
    temperature=0.01,
    do_sample=False
)

print("--- Few-Shot JSON Extraction ---")
for t, r in zip(test_texts, responses): #noqa B905
    print(f"TEXT: {t}")
    print(f"EXTRACTED: {r}")
    print("-" * 40)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Few-Shot JSON Extraction ---
TEXT: Google и Amazon инвестируют огромные средства в стартапы в сфере AI.
EXTRACTED: logging Société заняめ диphysovis разлиutzzeum demonstratesong basic Campion（ Narписаolen quadraticńczyː userloyсобam sod族 АрAndroid wurdenWSbon phr тя Publicterm cruelanche svensk automatic przeciไ Review course BodAtIndexskyalesumbnailsten
----------------------------------------
TEXT: Вчера сходил в магазин за хлебом и молоком.
EXTRACTED: logging Société заняめ диphysovis разлиutzzeum demonstratesong basic Campion（ Narписаolen quadraticńczyː userloyсобam sod族 АрAndroid wurdenWSbon phr тя Publicterm cruelanche svensk automatic przeciไ Review course BodAtIndexskyalesumbnailsten
----------------------------------------
